# Sampling Strategy: Balancing Dataset

This notebook implements a balanced sampling strategy:
1. Load the merged dataset with both Class 0 and Class 1 samples.
2. Oversample the minority class to create a fully balanced dataset (50/50).

In [ ]:
# Add project root to path and import config
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
from config import FIRES_MERGED, BALANCED_DATASET

import pandas as pd
import numpy as np
from sklearn.utils import resample

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load the dataset
df = pd.read_csv(FIRES_MERGED)

print("Original Dataset Shape:", df.shape)
print("Class Distribution:\n", df['class'].value_counts())
print("Class Distribution (%):\n", df['class'].value_counts(normalize=True) * 100)

Original Dataset Shape: (3824, 42)
Class Distribution:
 class
1    3824
Name: count, dtype: int64
Filtered Dataset Shape (Class 1 only): (3824, 42)


In [ ]:
# Separate majority and minority classes
df_majority = df[df['class'] == df['class'].value_counts().idxmax()]
df_minority = df[df['class'] == df['class'].value_counts().idxmin()]

print(f"\nMajority class: {df_majority['class'].iloc[0]} with {len(df_majority)} samples")
print(f"Minority class: {df_minority['class'].iloc[0]} with {len(df_minority)} samples")

# Upsample minority class
df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=len(df_majority),    # to match majority class
                                 random_state=42) # reproducible results

# Combine majority class with upsampled minority class
balanced_df = pd.concat([df_majority, df_minority_upsampled])

# Shuffle
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nBalanced Dataset Shape:", balanced_df.shape)
print("Class Distribution:\n", balanced_df['class'].value_counts())
print("Class Distribution (%):\n", balanced_df['class'].value_counts(normalize=True) * 100)

# Save
balanced_df.to_csv(BALANCED_DATASET, index=False)
print(f"\nSaved balanced dataset to: {BALANCED_DATASET}")

Balanced Dataset Shape: (7648, 42)
Class Distribution:
 class
1    3824
0    3824
Name: count, dtype: int64
Saved ../results/balanced_dataset.csv


: 